## Great Expectations Data Quality Test

create a clean notebook-style workflow that loads all 9 Olist CSV files, previews them, and prepares them for testing using Great Expectations.
This is the simple and scalable way used in many data engineering projects.

In [ ]:
pip install great_expectations

In [2]:
import pandas as pd
import great_expectations as gx
from great_expectations import expectations as gxe

context = gx.get_context()

E0000 00:00:1773913232.166480    1929 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773913232.167723    1929 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1773913232.167728    1929 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1773913232.167730    1929 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1773913232.167731    1929 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [3]:

source_folder = "data/"
data_source_name = "olist_sale"

In [4]:
data_source = context.data_sources.add_pandas_filesystem(
    name=data_source_name, 
    base_directory=source_folder
)

In [5]:
# Get your pre-registered CSV datasource

data_source = context.data_sources.get(data_source_name)

In [6]:
asset_name = "olist_csv_files"

In [7]:
csv_files = {
    "customers_csv": "olist_customers_dataset.csv",
    "geolocation_csv": "olist_geolocation_dataset.csv",
    "order_items_csv": "olist_order_items_dataset.csv",
    "order_payments_csv": "olist_order_payments_dataset.csv",
    "order_reviews_csv": "olist_order_reviews_dataset.csv",
    "orders_csv": "olist_orders_dataset.csv",
    "products_csv": "olist_products_dataset.csv",
    "sellers_csv": "olist_sellers_dataset.csv",
    "category_translation_csv": "product_category_name_translation.csv"
}

In [8]:
assets = {}

for asset_name in csv_files.keys():
    try:
        # Try to create the asset
        assets[asset_name] = data_source.add_csv_asset(name=asset_name)
    except Exception:
        # If it already exists, retrieve it
        assets[asset_name] = context.data_sources.get(data_source_name).get_asset(asset_name)

In [9]:
# check existing data sources
print(context.data_sources.all())

{'olist_sale': PandasFilesystemDatasource(type='pandas_filesystem', name='olist_sale', id=UUID('5b833a16-4bbc-43b5-a442-c650886b0811'), assets=[CSVAsset(name='customers_csv', type='csv', id=UUID('89512dae-511b-4153-8101-2d81074ee18d'), order_by=[], batch_metadata={}, batch_definitions=[], connect_options={}, sep=None, delimiter=None, header='infer', names=None, index_col=None, usecols=None, dtype=None, engine=None, true_values=None, false_values=None, skipinitialspace=False, skiprows=None, skipfooter=0, nrows=None, na_values=None, keep_default_na=True, na_filter=True, verbose=None, skip_blank_lines=True, parse_dates=None, infer_datetime_format=None, keep_date_col=None, date_format=None, dayfirst=False, cache_dates=True, iterator=False, chunksize=None, compression='infer', thousands=None, decimal='.', lineterminator=None, quotechar='"', quoting=0, doublequote=True, escapechar=None, comment=None, encoding=None, encoding_errors='strict', dialect=None, on_bad_lines='error', delim_whitespac

In [10]:
batch_definition_name = "olist_orders_dataset.csv"
batch_definition_path = "olist_orders_dataset.csv"

batch_definition = assets[asset_name].add_batch_definition_path(
    name=batch_definition_name, path=batch_definition_path
)

In [11]:
batch = batch_definition.get_batch()

In [12]:
print(batch.head(4))

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 149.25it/s]

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   
2          2018-08-08 13:50:00           2018-08-17 18:06:29   
3          2017-11-22 13:39:59           2017-12-02 00:28

In [13]:
preset_expectation = gx.expectations.ExpectColumnMaxToBeBetween(
    column="order_purchase_timestamp", min_value="2016-01-01", max_value="2020-12-31"
)

In [14]:
validation_results = batch.validate(preset_expectation)

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 402.88it/s] 


In [15]:
print(validation_results)

{
  "expectation_config": {
    "severity": "critical",
    "type": "expect_column_max_to_be_between",
    "kwargs": {
      "batch_id": "olist_sale-category_translation_csv",
      "column": "order_purchase_timestamp",
      "min_value": "2016-01-01",
      "max_value": "2020-12-31"
    },
    "meta": {}
  },
  "success": true,
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  },
  "result": {
    "observed_value": "2018-10-17"
  },
  "meta": {}
}


In [16]:
suite_name = "temp_suite"

suite = gx.ExpectationSuite(name=suite_name)

suite = context.suites.add(suite)

In [17]:
suite.add_expectation(preset_expectation)

ExpectColumnMaxToBeBetween(id='ea43f649-39c1-4fd6-a89f-3450bd7d8e7e', meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=False, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_purchase_timestamp', row_condition=None, condition_parser=None, min_value=datetime.date(2016, 1, 1), max_value=datetime.date(2020, 12, 31), strict_min=False, strict_max=False)

In [18]:
definition_name = "validate_definition"
validation_definition = gx.ValidationDefinition(
    data=batch_definition, suite=suite, name=definition_name
)

In [19]:
validation_results = validation_definition.run()

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 374.42it/s] 


In [20]:
print(validation_results)

{
  "success": true,
  "id": null,
  "suite_parameters": {},
  "results": [
    {
      "expectation_config": {
        "id": "ea43f649-39c1-4fd6-a89f-3450bd7d8e7e",
        "severity": "critical",
        "type": "expect_column_max_to_be_between",
        "kwargs": {
          "batch_id": "olist_sale-category_translation_csv",
          "column": "order_purchase_timestamp",
          "min_value": "2016-01-01",
          "max_value": "2020-12-31"
        },
        "meta": {}
      },
      "success": true,
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      },
      "result": {
        "observed_value": "2018-10-17"
      },
      "meta": {}
    }
  ],
  "statistics": {
    "evaluated_expectations": 1,
    "successful_expectations": 1,
    "unsuccessful_expectations": 0,
    "success_percent": 100.0
  },
  "suite_name": "temp_suite",
  "meta": {
    "great_expectations_version": "1.8.0",
    "batch_s

In [20]:
import pandas as pd
import great_expectations as gx
from great_expectations import expectations as gxe

for asset_name, file_name in csv_files.items():

    # Read CSV directly
    df = pd.read_csv(f"data/{file_name}")

    print(f"\nPreview of {asset_name}")
    print(df.head(5))


Preview of customers_csv
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

Preview of geolocation_csv
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037 

In [21]:
import pandas as pd
import great_expectations as gx
from great_expectations import expectations as gxe

In [22]:
orders_data = pd.read_csv("data/olist_orders_dataset.csv")

In [61]:
# Get context
context = gx.get_context()

# Create datasource
data_source_name = "olist_dataframe"
data_source = context.data_sources.add_pandas(name=data_source_name)

# Create asset
data_asset_name = "olist_orders_asset"
data_asset = data_source.add_dataframe_asset(name=data_asset_name)

# Create batch definition
batch_definition_name = "olist_orders_dataframe"
batch_definition = data_asset.add_batch_definition_whole_dataframe(
    batch_definition_name
)

# Pass dataframe as batch
batch_parameters = {"dataframe": orders_data}

# Get batch
new_batch = batch_definition.get_batch(batch_parameters=batch_parameters)

In [24]:
print(new_batch.head(10))

Calculating Metrics: 100%|██████████| 1/1 [00:00<00:00, 177.16it/s]

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
5  a4591c265e18cb1dcee52889e2d8acc3  503740e9ca751ccdda7ba28e9ab8f608   
6  136cce7faa42fdb2cefd53fdc79a6098  ed0271e0b7da060a393796590e7b737a   
7  6514b8ad8028c9f2cc2374ded245783f  9bdf08b4b3b52b5526ff42d37d47f222   
8  76c6e866289321a7c93b82b54852dc33  f54a9f0e6b351c431402b8461ea51999   
9  e69bfb5eb88e0ed6a785585b27e16dbf  31ad1d1b63eb9962463f764d4e6e0c9d   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2   

In [25]:
# we can create new expectations 
new_expectation = gx.expectations.ExpectColumnToExist(column="order_status",column_index=2
)

In [26]:
validation_results = new_batch.validate(new_expectation)

Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 508.68it/s]


In [27]:
print(validation_results)

{
  "success": true,
  "meta": {},
  "expectation_config": {
    "meta": {},
    "type": "expect_column_to_exist",
    "severity": "critical",
    "kwargs": {
      "batch_id": "olist_dataframe-olist_orders_asset",
      "column": "order_status",
      "column_index": 2
    }
  },
  "result": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}


In [28]:
# Null checks
new_expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
    column=("order_status")
)


In [29]:
columns_to_check = [
     "customer_id",
     "order_purchase_timestamp"
]

for col in columns_to_check:
    new_expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
        column=col
    )
    suite.add_expectation(new_expectation)

In [30]:
validator = context.get_validator(
    batch=new_batch,
    expectation_suite=suite
)

In [31]:
validation_results = validator.validate()

print(validation_results)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 309.72it/s]

{
  "success": true,
  "results": [
    {
      "success": true,
      "meta": {},
      "expectation_config": {
        "id": "3963c51d-256f-436f-a32c-b79034e72d7d",
        "meta": {},
        "type": "expect_column_max_to_be_between",
        "severity": "critical",
        "kwargs": {
          "batch_id": "olist_dataframe-olist_orders_asset",
          "column": "order_purchase_timestamp",
          "min_value": "2016-01-01",
          "max_value": "2020-12-31"
        }
      },
      "result": {
        "observed_value": "2018-10-17"
      },
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "meta": {},
      "expectation_config": {
        "meta": {},
        "type": "expect_column_values_to_not_be_null",
        "severity": "critical",
        "kwargs": {
          "batch_id": "olist_dataframe-olist_orders_asset",
          "column": "order_purcha

In [32]:
# Duplicate checks
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="order_id"
    )
)

ExpectColumnValuesToBeUnique(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_id', mostly=1, row_condition=None, condition_parser=None)

In [33]:
validation_results = validator.validate()
print(validation_results)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 368.19it/s]

{
  "success": true,
  "results": [
    {
      "success": true,
      "meta": {},
      "expectation_config": {
        "id": "3963c51d-256f-436f-a32c-b79034e72d7d",
        "meta": {},
        "type": "expect_column_max_to_be_between",
        "severity": "critical",
        "kwargs": {
          "batch_id": "olist_dataframe-olist_orders_asset",
          "column": "order_purchase_timestamp",
          "min_value": "2016-01-01",
          "max_value": "2020-12-31"
        }
      },
      "result": {
        "observed_value": "2018-10-17"
      },
      "exception_info": {
        "raised_exception": false,
        "exception_traceback": null,
        "exception_message": null
      }
    },
    {
      "success": true,
      "meta": {},
      "expectation_config": {
        "meta": {},
        "type": "expect_column_values_to_not_be_null",
        "severity": "critical",
        "kwargs": {
          "batch_id": "olist_dataframe-olist_orders_asset",
          "column": "order_purcha

In [34]:
print(validation_results.success)

True


In [35]:
validation_results["results"]

[{
   "success": true,
   "meta": {},
   "expectation_config": {
     "id": "3963c51d-256f-436f-a32c-b79034e72d7d",
     "meta": {},
     "type": "expect_column_max_to_be_between",
     "severity": "critical",
     "kwargs": {
       "batch_id": "olist_dataframe-olist_orders_asset",
       "column": "order_purchase_timestamp",
       "min_value": "2016-01-01",
       "max_value": "2020-12-31"
     }
   },
   "result": {
     "observed_value": "2018-10-17"
   },
   "exception_info": {
     "raised_exception": false,
     "exception_traceback": null,
     "exception_message": null
   }
 },
 {
   "success": true,
   "meta": {},
   "expectation_config": {
     "meta": {},
     "type": "expect_column_values_to_not_be_null",
     "severity": "critical",
     "kwargs": {
       "batch_id": "olist_dataframe-olist_orders_asset",
       "column": "order_purchase_timestamp"
     }
   },
   "result": {
     "element_count": 99441,
     "unexpected_count": 0,
     "unexpected_percent": 0.0,
     "p

In [36]:
orders_data["order_id"].duplicated().sum()

np.int64(0)

In [37]:
#valid order status values
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="order_status",
        value_set=[
            "created",
            "approved",
            "invoiced",
            "processing",
            "shipped",
            "delivered",
            "canceled"
        ]
    )
)

ExpectColumnValuesToBeInSet(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_status', mostly=1, row_condition=None, condition_parser=None, value_set=['created', 'approved', 'invoiced', 'processing', 'shipped', 'delivered', 'canceled'])

In [38]:
# Date range validation
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="order_purchase_timestamp",
        min_value="2016-01-01",
        max_value="2018-12-31"
    )
)

ExpectColumnValuesToBeBetween(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='order_purchase_timestamp', mostly=1, row_condition=None, condition_parser=None, min_value=datetime.date(2016, 1, 1), max_value=datetime.date(2018, 12, 31), strict_min=False, strict_max=False)

In [39]:
# Numeric Checks (for order items table)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0
    )
)

ExpectColumnValuesToBeBetween(id=None, meta=None, notes=None, result_format=<ResultFormat.BASIC: 'BASIC'>, description=None, catch_exceptions=True, rendered_content=None, severity=<FailureSeverity.CRITICAL: 'critical'>, windows=None, batch_id=None, column='price', mostly=1, row_condition=None, condition_parser=None, min_value=0.0, max_value=None, strict_min=False, strict_max=False)

In [40]:
# validation for the above parts
validation_results = validator.validate()

print(validation_results.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 315.10it/s]

True


In [41]:
# Sanity check similar to what Great Expectation does  but in pandas
import pandas as pd

# Columns to check
columns_to_check = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

# Check for nulls
nulls = orders_data[columns_to_check].isnull().sum()

# Check for duplicates (only for order_id)
duplicates = orders_data["order_id"].duplicated().sum()

# Optional: check order_status values
valid_statuses = ["created", "approved", "invoiced", "processing", "shipped", "delivered", "canceled"]
invalid_statuses = orders_data.loc[~orders_data["order_status"].isin(valid_statuses), "order_status"].unique()

# Summary output
print("Null counts per column:\n", nulls)
print("\nNumber of duplicate order_ids:", duplicates)
print("\nInvalid order_status values:", invalid_statuses)


Null counts per column:
 order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_estimated_delivery_date    0
dtype: int64

Number of duplicate order_ids: 0

Invalid order_status values: ['unavailable']


In [42]:
# set up for all tables GX
import great_expectations as gx

context = gx.get_context()

In [43]:
# correct one table at a time which get false/ Customers data
import pandas as pd

# Load CSV
customers_df = pd.read_csv("data/olist_customers_dataset.csv")

#  Check for nulls
print(customers_df.isnull().sum())

#  Drop rows with nulls
customers_df = customers_df.dropna(subset=["customer_id", "customer_unique_id"])

#  Drop duplicates
customers_df = customers_df.drop_duplicates(subset=["customer_unique_id"])

# Save cleaned version if needed
customers_df.to_csv("data/olist_customers_clean.csv", index=False)

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [44]:
# order_items data cleaning
import pandas as pd

# Load CSV
order_items_df = pd.read_csv("data/olist_order_items_dataset.csv")

#  Check for nulls
print("Nulls in each column:\n", order_items_df.isnull().sum())

# Drop rows with nulls in critical columns
order_items_df = order_items_df.dropna(subset=["order_id", "product_id"])

# 3Drop duplicates based on order_id + product_id (composite key)
order_items_df = order_items_df.drop_duplicates(subset=["order_id", "product_id"])

#  Save cleaned version
order_items_df.to_csv("data/olist_order_items_clean.csv", index=False)

print("\nCleaned order_items dataset saved as 'olist_order_items_clean.csv'")

Nulls in each column:
 order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Cleaned order_items dataset saved as 'olist_order_items_clean.csv'


In [45]:
# clean payment csv
import pandas as pd

# Load CSV
order_payments_df = pd.read_csv("data/olist_order_payments_dataset.csv")

#  Check for nulls
print("Nulls in each column:\n", order_payments_df.isnull().sum())

# Drop rows with nulls in critical columns
order_payments_df = order_payments_df.dropna(subset=["order_id", "payment_type", "payment_value"])

# Drop duplicates based on order_id + payment_type
order_payments_df = order_payments_df.drop_duplicates(subset=["order_id", "payment_type"])

#  Save cleaned version
order_payments_df.to_csv("data/olist_order_payments_clean.csv", index=False)

print("\nCleaned order_payments dataset saved as 'olist_order_payments_clean.csv'")

Nulls in each column:
 order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Cleaned order_payments dataset saved as 'olist_order_payments_clean.csv'


In [46]:
# clean order reviews
import pandas as pd

# Load CSV
order_reviews_df = pd.read_csv("data/olist_order_reviews_dataset.csv")

# Check for nulls
print("Nulls in each column:\n", order_reviews_df.isnull().sum())

# Drop rows with nulls in critical columns
#critical_cols = ["review_id", "order_id", "review_score"]
order_reviews_df = order_reviews_df.dropna(subset=["review_id", "order_id", "review_score"])

# Drop duplicates based on review_id
order_reviews_df = order_reviews_df.drop_duplicates(subset=["review_id"])

# Save cleaned version
order_reviews_df.to_csv("data/olist_order_reviews_clean.csv", index=False)

print("\nCleaned order_reviews dataset saved as 'olist_order_reviews_clean.csv'")

Nulls in each column:
 review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Cleaned order_reviews dataset saved as 'olist_order_reviews_clean.csv'


In [47]:
# clean orders csv
import pandas as pd

# Load CSV
orders_df = pd.read_csv(
    "data/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

# Check for nulls
print("Nulls in each column:\n", orders_df.isnull().sum())

#  Drop rows with nulls in critical columns
#ritical_cols = ["order_id", "customer_id", "order_status", "order_purchase_timestamp"]
orders_df = orders_df.dropna(subset=["order_id", "customer_id", "order_status", "order_purchase_timestamp"])

# Keep only valid order_status
allowed_status = ["created","approved","invoiced","processing","shipped","delivered","canceled"]
orders_df = orders_df[orders_df["order_status"].isin(allowed_status)]

#  Drop duplicate order_ids
orders_df = orders_df.drop_duplicates(subset=["order_id"])

# Save cleaned version
orders_df.to_csv("data/olist_orders_clean.csv", index=False)

print("\nCleaned orders dataset saved as 'olist_orders_clean.csv'")

Nulls in each column:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Cleaned orders dataset saved as 'olist_orders_clean.csv'


In [51]:
# clean procuts table
import pandas as pd

# Load CSV
products_df = pd.read_csv("data/olist_products_dataset.csv")

# Check for nulls
print("Nulls in each column:\n", products_df.isnull().sum())

# Drop rows with nulls in critical columns
#critical_cols = ["product_id"]
products_df = products_df.dropna(subset=["product_id"])

# Drop duplicate product_ids
products_df = products_df.drop_duplicates(subset=["product_id"])

#  Save cleaned version
products_df.to_csv("data/olist_products_clean.csv", index=False)

print("\nCleaned products dataset saved as 'olist_products_clean.csv'")


Nulls in each column:
 product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Cleaned products dataset saved as 'olist_products_clean.csv'


In [53]:
ccsv_files = {
    "customers_csv": "olist_customers_clean.csv",
    "geolocation_csv": "olist_geolocation_dataset.csv",
    "order_items_csv": "olist_order_items_clean.csv",
    "order_payments_csv": "olist_order_payments_clean.csv",
    "order_reviews_csv": "olist_order_reviews_clean.csv",
    "orders_csv": "olist_orders_clean.csv",
    "products_csv": "olist_products_clean.csv",
    "sellers_csv": "olist_sellers_dataset.csv",
    "category_translation_csv": "product_category_name_translation.csv"
}